In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
# load_dotenv(Path.cwd() / ".env")
load_dotenv()

True

In [ ]:
SERVICENOW_USERNAME=os.getenv("SERVICENOW_USERNAME")
SERVICENOW_PASSWORD=os.getenv("SERVICENOW_PASSWORD")
INSTANCE=os.getenv("SERVICENOW_INSTANCE")

```
{
  "short_description": "VPN not working",
  "description": "User unable to connect to VPN",
  "caller_id": "admin",
  "category": "network",
  "subcategory": "vpn",
  "impact": "2",
  "urgency": "2",
  "priority": "2",
  "assignment_group": "Network",
  "assigned_to": "admin",
  "cmdb_ci": "VPN Gateway",
  "comments": "Created from API",
  "work_notes": "Investigating issue"
}
```

In [ ]:
import requests

# INSTANCE = "https://dev397466.service-now.com"
# USERNAME = "your_username"
# PASSWORD = "your_password"

url = f"{INSTANCE}/api/now/table/incident"

payload = {
    "short_description": "Test incident from Python",
    "description": "Created using ServiceNow REST API",
    "urgency": "2",
    "impact": "2",
}

response = requests.post(
    url,
    auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    },
    json=payload,
)

response.raise_for_status()
print(response.json())

{'result': {'parent': '', 'made_sla': 'true', 'watch_list': '', 'upon_reject': 'cancel', 'sys_updated_on': '2026-06-03 09:10:54', 'child_incidents': '0', 'hold_reason': '', 'origin_table': '', 'task_effective_number': 'INC0010003', 'approval_history': '', 'number': 'INC0010003', 'resolved_by': '', 'sys_updated_by': 'admin', 'opened_by': {'link': 'https://dev397466.service-now.com/api/now/table/sys_user/6816f79cc0a8016401c5a33be04be441', 'value': '6816f79cc0a8016401c5a33be04be441'}, 'user_input': '', 'sys_created_on': '2026-06-03 09:10:54', 'sys_domain': {'link': 'https://dev397466.service-now.com/api/now/table/sys_user_group/global', 'value': 'global'}, 'state': '1', 'route_reason': '', 'sys_created_by': 'admin', 'knowledge': 'false', 'order': '', 'calendar_stc': '', 'closed_at': '', 'cmdb_ci': '', 'delivery_plan': '', 'contract': '', 'impact': '2', 'active': 'true', 'work_notes_list': '', 'business_service': '', 'business_impact': '', 'priority': '3', 'sys_domain_path': '/', 'time_wor

In [5]:
response.json()['result']['opened_by']['link']

'https://dev397466.service-now.com/api/now/table/sys_user/6816f79cc0a8016401c5a33be04be441'

In [6]:
response.json()['result']['number']

'INC0010003'

#### Get Historical Incidents

In [20]:
import requests

INSTANCE = "https://dev397466.service-now.com"

response = requests.get(
    f"{INSTANCE}/api/now/table/incident",
    auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
    params={
        "sysparm_limit": 100,
        "sysparm_display_value": "true"
    }
)

incidents = response.json()["result"]

for incident in incidents:
    print(
        incident["number"],
        incident.get("short_description"),
        incident.get("close_notes")
    )

In [21]:
incidents

[]

In [22]:
import requests

INSTANCE = "https://dev397466.service-now.com"

incident_number = "INC0010019"

response = requests.get(
    f"{INSTANCE}/api/now/table/incident",
    auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
    params={
        "sysparm_query": f"number={incident_number}",
        "sysparm_display_value": "true"
    }
)

incident = response.json()["result"][0]

for k, v in incident.items():
    print(f"{k}: {v}")

parent: 
made_sla: true
watch_list: 
upon_reject: Cancel all future Tasks
sys_updated_on: 2026-06-17 06:10:50
child_incidents: 0
hold_reason: 
origin_table: 
task_effective_number: INC0010019
approval_history: 
number: INC0010019
resolved_by: {'display_value': 'System Administrator', 'link': 'https://dev397466.service-now.com/api/now/table/sys_user/6816f79cc0a8016401c5a33be04be441'}
sys_updated_by: admin
opened_by: {'display_value': 'System Administrator', 'link': 'https://dev397466.service-now.com/api/now/table/sys_user/6816f79cc0a8016401c5a33be04be441'}
user_input: 
sys_created_on: 2026-06-17 05:45:41
sys_domain: {'display_value': 'global', 'link': 'https://dev397466.service-now.com/api/now/table/sys_user_group/global'}
state: Resolved
route_reason: 
sys_created_by: admin
knowledge: false
order: 
calendar_stc: 1,509
closed_at: 
cmdb_ci: 
delivery_plan: 
contract: 
impact: 2 - Medium
active: true
work_notes_list: 
business_service: 
business_impact: 
priority: 3 - Moderate
sys_domain_

In [25]:

# print('short description: ',incident['short_description'])
# print('short description: ',incident['opened_at'])
print("DETAILS FOR THE INCIDENT NUMBER:",incident_number)
print(f"short description: { incident['short_description'] }\
      \nopened_at: { incident['opened_at'] }\
      \nseverity: { incident['severity'] } \
      \nincident_state: { incident['incident_state'] }\
      \nclose_notes: { incident['close_notes'] } ")


DETAILS FOR THE INCIDENT NUMBER: INC0010019
short description: Database connectivity outage for app abc-dev1 — my-db-dev.com:5432      
opened_at: 2026-06-17 05:45:41      
severity: 3 - Low       
incident_state: Resolved      
close_notes: my-db-dev.com:5432 this is old DNS you need to use the new DNS my-db2-dev.com:5432 


In [19]:
import requests

INSTANCE = "https://dev397466.service-now.com"

# WARNING: This deletes every incident the current ServiceNow user can delete.
# Use only in a disposable POC/dev instance.
batch_size = 100
deleted_count = 0

while True:
    response = requests.get(
        f"{INSTANCE}/api/now/table/incident",
        auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
        params={
            "sysparm_fields": "number,sys_id,short_description",
            "sysparm_limit": batch_size,
        },
    )
    response.raise_for_status()

    incidents_to_delete = response.json()["result"]

    if not incidents_to_delete:
        break

    for incident in incidents_to_delete:
        delete_response = requests.delete(
            f"{INSTANCE}/api/now/table/incident/{incident['sys_id']}",
            auth=(SERVICENOW_USERNAME, SERVICENOW_PASSWORD),
            headers={"Accept": "application/json"},
        )

        if delete_response.status_code in (200, 204):
            deleted_count += 1
            print(f"Deleted {incident['number']} - {incident['short_description']}")
        else:
            print(
                f"Failed to delete {incident['number']}: "
                f"{delete_response.status_code} {delete_response.text}"
            )

print(f"Deleted {deleted_count} incidents.")

Deleted INC0000060 - Unable to connect to email
Deleted INC0009002 - My computer is not detecting the headphone device
Deleted INC0000009 - Reset my password
Deleted INC0000010 - Need Oracle 10GR2 installed
Deleted INC0000011 - Need new Blackberry set up
Deleted INC0000012 - Customer didn't receive eFax
Deleted INC0000013 - EMAIL is slow when an attachment is involved
Deleted INC0000014 - Missing my home directory
Deleted INC0000021 - New employee hire
Deleted INC0000024 - Issue with a web page on wiki
Deleted INC0000026 - Seem to have an issue with my hard drive...
Deleted INC0000028 - My disk is still having issues. Can't delete a file
Deleted INC0000030 - Lost connection to the wireless network
Deleted INC0000032 - EMAIL Server Down Again
Deleted INC0000033 - File Server is 80% full - Needs upgrade
Deleted INC0000034 - Does not look like a backup occurred last night
Deleted INC0000035 - Reset my password
Deleted INC0000036 - Issue with networking
Deleted INC0000038 - my PDF docs are